# Software Engineering - Tutorial 1: UML Modeling
## Assignment 1 - Group 2
**Instructor:** PhD. Manh-Hung Ha  
**Repository Directory:** `D:\Job\thothan\VNU\Software Engineering\Assigment1`  
**Master Draw.io File:** [`assigment1_all_diagrams.drawio`](assigment1_all_diagrams.drawio) (Contains 3 multi-page tabs for StarUML / Diagrams.net)

---
### 📌 Executive Summary of Assignment 1
This notebook provides the complete theoretical, architectural, and executable solutions for all 3 exercises in **Tutorial 1 - UML Modeling**:
1. **Exercise 1: Adapter Design Pattern** - Unified rendering system for geometric shapes supporting PDF, PNG, and SVG (integrating an incompatible external `SVGWriter` library). Includes Class Diagram & Sequence Diagram.
2. **Exercise 2: Composite Design Pattern** - Hierarchical tree structure manipulation with abstract `Element`, composite `Noeud`, and leaf `Feuille`. Includes Class Diagram, Object Diagram (Figure 4), Sequence Diagram for recursive traversal, and algorithmic complexity/traversal type analysis.
3. **Exercise 3: Authentication System** - Complete design of a secure multi-user credential verification workflow with `IULogin`, `GestionnairePersonne`, and `Personne`. Includes Class Diagram, Activity Diagram, executable Python implementation, and Object Diagram during execution.

---
# Exercise 1: Geometric Shapes & Adapter Pattern

### 1. Problem Statement
We have an object-oriented drawing application that manipulates geometric shapes (`Circle`, `Rectangle`, `Line`, etc.) through an abstract rendering interface `MoteurRendu`.
- Existing renderers: `MoteurRenduPDF` and `MoteurRenduPNG` are already implemented.
- Objective: Export shapes to the SVG file format using an external third-party library `SVGWriter`.
- Technical Challenge: `SVGWriter` provides an incompatible interface:
  - Expected (`MoteurRendu`): `dessinerCercle(x, y, rayon)`, `dessineRectangle(x, y, longueur, largeur)`, `dessinerLigne(x1, y1, x2, y2)`.
  - Provided (`SVGWriter`): `addCercle(cx, cy, r)`, `addRectangle(x, y, longueur, largeur)`, `addLigne(x1, y1, x2, y2)`.

---
### 2. Theoretical Analysis & Design Pattern
We apply the **Adapter Pattern (Object Adapter)**:
- **Target Interface (`MoteurRendu`)**: Defines the domain-specific drawing operations used by the client `Forme`.
- **Adaptee (`SVGWriter`)**: The existing third-party class that provides the actual SVG rendering logic but has different method signatures.
- **Adapter (`AdaptateurSVG`)**: Implements `MoteurRendu` and maintains an instance of `SVGWriter` (composition `1 -writer`). When a target method is invoked (e.g., `dessineRectangle`), the adapter translates and delegates the call to the adaptee method (`writer.addRectangle`).

---
### 3. UML Diagrams
- Draw.io Source: [`exercise1/exercise1_diagrams.drawio`](exercise1/exercise1_diagrams.drawio)

#### Question 1: Class Diagram (Adapter Pattern)
![Exercise 1 Class Diagram](diagram_images/ex1_class_diagram.png)

#### Question 2: Sequence Diagram (Exporting Rectangle to SVG)
![Exercise 1 Sequence Diagram](diagram_images/ex1_sequence_diagram.png)

**Sequence of Messages:**
1. Caller (`main`) invokes `r.dessiner(moteurSVG)` on `r : Rectangle`.
2. `r : Rectangle` invokes `moteurSVG.dessineRectangle(20, 30, 200, 100)` on `moteurSVG : AdaptateurSVG`.
3. `moteurSVG : AdaptateurSVG` adapts and delegates to `writer.addRectangle(20, 30, 200, 100)` on `writer : SVGWriter`.
4. `writer : SVGWriter` outputs the SVG XML tag: `<rect x="20" y="30" width="200" height="100"/>`.
5. Returns propagate back to `r` and then to `main`.

In [1]:
"""
Exercise 1: Adapter Pattern for SVG Drawing Engine
Tutorial 1 - UML Modeling (Software Engineering)
"""
from abc import ABC, abstractmethod


# =====================================================================
# Target Interface & Concrete Renderers
# =====================================================================
class MoteurRendu(ABC):
    """Target Abstract Interface for Rendering Shapes."""

    @abstractmethod
    def dessinerCercle(self, x: float, y: float, rayon: float) -> None:
        pass

    @abstractmethod
    def dessineRectangle(self, x: float, y: float, longueur: float, largeur: float) -> None:
        pass

    @abstractmethod
    def dessinerLigne(self, x1: float, y1: float, x2: float, y2: float) -> None:
        pass


class MoteurRenduPDF(MoteurRendu):
    """Concrete PDF Renderer."""

    def dessinerCercle(self, x: float, y: float, rayon: float) -> None:
        print("PDF : cercle")

    def dessineRectangle(self, x: float, y: float, longueur: float, largeur: float) -> None:
        print("PDF : rectangle")

    def dessinerLigne(self, x1: float, y1: float, x2: float, y2: float) -> None:
        print("PDF : ligne")


class MoteurRenduPNG(MoteurRendu):
    """Concrete PNG Renderer."""

    def dessinerCercle(self, x: float, y: float, rayon: float) -> None:
        print("PNG : cercle")

    def dessineRectangle(self, x: float, y: float, longueur: float, largeur: float) -> None:
        print("PNG : rectangle")

    def dessinerLigne(self, x1: float, y1: float, x2: float, y2: float) -> None:
        print("PNG : ligne")


# =====================================================================
# Adaptee (External Library)
# =====================================================================
class SVGWriter:
    """External Library with incompatible interface for SVG export."""

    def addCercle(self, cx: float, cy: float, r: float) -> None:
        print(f'<circle cx="{int(cx)}" cy="{int(cy)}" r="{int(r)}"/>')

    def addRectangle(self, x: float, y: float, longueur: float, largeur: float) -> None:
        print(f'<rect x="{int(x)}" y="{int(y)}" width="{int(longueur)}" height="{int(largeur)}"/>')

    def addLigne(self, x1: float, y1: float, x2: float, y2: float) -> None:
        print(f'<line x1="{int(x1)}" y1="{int(y1)}" x2="{int(x2)}" y2="{int(y2)}" stroke="black" stroke-width="2"/>')


# =====================================================================
# Adapter (Object Adapter Pattern)
# =====================================================================
class AdaptateurSVG(MoteurRendu):
    """Adapter bridging MoteurRendu calls to SVGWriter methods."""

    def __init__(self, writer: SVGWriter | None = None) -> None:
        self.writer = writer if writer is not None else SVGWriter()

    def dessinerCercle(self, x: float, y: float, rayon: float) -> None:
        self.writer.addCercle(x, y, rayon)

    def dessineRectangle(self, x: float, y: float, longueur: float, largeur: float) -> None:
        self.writer.addRectangle(x, y, longueur, largeur)

    def dessinerLigne(self, x1: float, y1: float, x2: float, y2: float) -> None:
        self.writer.addLigne(x1, y1, x2, y2)


# =====================================================================
# Geometric Shapes Hierarchy
# =====================================================================
class Forme(ABC):
    """Base class for geometric shapes."""

    @abstractmethod
    def dessiner(self, moteur: MoteurRendu) -> None:
        pass


class Cercle(Forme):
    def __init__(self, x: float, y: float, rayon: float) -> None:
        self.x = x
        self.y = y
        self.rayon = rayon

    def dessiner(self, moteur: MoteurRendu) -> None:
        moteur.dessinerCercle(self.x, self.y, self.rayon)


class Rectangle(Forme):
    def __init__(self, x: float, y: float, longueur: float, largeur: float) -> None:
        self.x = x
        self.y = y
        self.longueur = longueur
        self.largeur = largeur

    def dessiner(self, moteur: MoteurRendu) -> None:
        moteur.dessineRectangle(self.x, self.y, self.longueur, self.largeur)


class Ligne(Forme):
    def __init__(self, x1: float, y1: float, x2: float, y2: float) -> None:
        self.x1 = x1
        self.y1 = y1
        self.x2 = x2
        self.y2 = y2

    def dessiner(self, moteur: MoteurRendu) -> None:
        moteur.dessinerLigne(self.x1, self.y1, self.x2, self.y2)


# =====================================================================
# Main execution matching tutorial requirements
# =====================================================================
def main():
    c = Cercle(100, 100, 50)
    r = Rectangle(20, 30, 200, 100)
    l = Ligne(10, 10, 20, 20)

    moteurPNG = MoteurRenduPNG()
    moteurPDF = MoteurRenduPDF()
    moteurSVG = AdaptateurSVG()

    print("===== PDF =====")
    c.dessiner(moteurPDF)
    r.dessiner(moteurPDF)
    l.dessiner(moteurPDF)

    print("\n===== PNG =====")
    c.dessiner(moteurPNG)
    r.dessiner(moteurPNG)
    l.dessiner(moteurPNG)

    print("\n===== SVG =====")
    c.dessiner(moteurSVG)
    r.dessiner(moteurSVG)
    l.dessiner(moteurSVG)


if __name__ == "__main__":
    main()

===== PDF =====
PDF : cercle
PDF : rectangle
PDF : ligne

===== PNG =====
PNG : cercle
PNG : rectangle
PNG : ligne

===== SVG =====
<circle cx="100" cy="100" r="50"/>
<rect x="20" y="30" width="200" height="100"/>
<line x1="10" y1="10" x2="20" y2="20" stroke="black" stroke-width="2"/>


---
# Exercise 2: Composite Pattern (Element, Noeud, Feuille)

### 1. Problem Statement & Architecture
We model a hierarchical tree data structure using the **Composite Pattern**:
- **Component (`Element`)**: Abstract base class declaring the unified operation `afficheElement()`.
- **Leaf (`Feuille`)**: Represents terminal nodes storing a single integer `valeur`.
- **Composite (`Noeud`)**: Represents internal branch nodes containing a collection `lesElements` of `Element` children, supporting recursive traversal via `afficheElement()`.

---
### 2. Answers to Tutorial Questions

#### Question 1: Instantiation of Object Diagram (Figure 4)
- Leaf `f1` is instantiated with `valeur = 1`.
- Leaf `f2` is instantiated with `valeur = 2`.
- Leaf `f3` is instantiated with `valeur = 3`.
- Internal node `n2` contains children `[f2, f1]` via `n2.setElement(f2, f1)`.
- Root node `n1` contains children `[n2, f3]` via `n1.setElement(n2, f3)`.

#### Question 2: Class Diagram
See the left panel below for the inheritance relationships and composition (`Noeud` composed of `0..* Element`).

#### Question 3: Sequence Diagram for `n1.afficheElement()`
See the Sequence Diagram below illustrating activation lifetimes and message sequence.

#### Question 4: Displayed Output & Tree Traversal Type
- **Output displayed when executing `n2.afficheElement()`:**
  ```text
  2
  1
  ```
  *(Because `n2` was constructed with `n2.setElement(f2, f1)`, where `f2.valeur = 2` and `f1.valeur = 1`).*
- **Type of Traversal:** **Pre-order Traversal (Depth-First Search - DFS)**. In this recursive traversal, parent nodes delegate display to their child elements from left to right, visiting each branch completely before moving to adjacent siblings.

---
### 3. UML Diagrams
- Draw.io Source: [`exercise2/exercise2_diagrams.drawio`](exercise2/exercise2_diagrams.drawio)

#### Class Diagram & Object Diagram (Figure 4)
![Exercise 2 Class and Object Diagram](diagram_images/ex2_class_object_diagram.png)

#### Sequence Diagram: Execution of `n1.afficheElement()`
![Exercise 2 Sequence Diagram](diagram_images/ex2_sequence_diagram.png)

In [2]:
"""
Exercise 2: Composite Pattern (Element, Noeud, Feuille)
Tutorial 1 - UML Modeling (Software Engineering)
"""
from abc import ABC, abstractmethod
from typing import List


# =====================================================================
# Component Interface
# =====================================================================
class Element(ABC):
    """Abstract Base Class representing an Element in the tree structure."""

    @abstractmethod
    def afficheElement(self) -> None:
        """Display the element."""
        pass


# =====================================================================
# Leaf (Feuille)
# =====================================================================
class Feuille(Element):
    """Leaf class storing a single integer value."""

    def __init__(self, uneValeur: int) -> None:
        self.valeur = uneValeur

    def afficheElement(self) -> None:
        print(self.valeur)


# =====================================================================
# Composite Node (Noeud)
# =====================================================================
class Noeud(Element):
    """Composite class containing a collection of Element children."""

    def __init__(self) -> None:
        self.lesElements: List[Element] = []

    def setElement(self, e1: Element, e2: Element) -> None:
        """Adds child elements e1 and e2 to the elements list."""
        self.lesElements.append(e1)
        self.lesElements.append(e2)

    def afficheElement(self) -> None:
        """Recursively displays all child elements (Pre-order traversal)."""
        for elem in self.lesElements:
            elem.afficheElement()


# =====================================================================
# Main execution matching object diagram (Figure 4)
# =====================================================================
def main():
    # Instantiating the object diagram from Figure 4
    n1 = Noeud()
    n2 = Noeud()
    f1 = Feuille(1)
    f2 = Feuille(2)
    f3 = Feuille(3)

    # Establishing hierarchy:
    # n2 contains f2 and f1
    n2.setElement(f2, f1)

    # n1 contains n2 and f3
    n1.setElement(n2, f3)

    print("--- Executing n1.afficheElement() ---")
    n1.afficheElement()

    print("\n--- Executing n2.afficheElement() ---")
    n2.afficheElement()


if __name__ == "__main__":
    main()

--- Executing n1.afficheElement() ---
2
1
3

--- Executing n2.afficheElement() ---
2
1


---
# Exercise 3: User Authentication System

### 1. Problem Statement & Architecture
The application models a secure user identification procedure:
- **`Personne`**: Entity storing credentials (`nom`, `motDePasse`).
- **`GestionnairePersonne`**: Domain manager storing registered authorized persons (`lesPersonnes: Personne[*]`) and verifying credentials (`verifierMotDePasse`).
- **`IULogin`**: Boundary/Presentation class interacting with the user (`demanderNom`, `demanderMotDePasse`), triggering identification through `GestionnairePersonne`, and notifying results (`autoriserConnexion` or `interdireConnexion`).

---
### 2. Answers to Tutorial Questions

#### Question 1: Class Diagram
Modeled with unidirectional associations: `IULogin` maintains `1 -gestionnaire: GestionnairePersonne`, and `GestionnairePersonne` aggregates `* -lesPersonnes: Personne`.

#### Question 2: Activity Diagram of `IULogin.identification()`
Models the exact execution logic:
- `demanderNom()` ➔ `demanderMotDePasse()` ➔ Call `gestionnaire.verifierMotDePasse(nom, motDePasse)` ➔ Decision diamond:
  - If `[res == true]` ➔ `autoriserConnexion()` ➔ End.
  - If `[res == false]` ➔ `interdireConnexion()` ➔ End.

#### Question 3: Implementation & Validation
Completed `identification()` and `verifierMotDePasse()`, verified with both valid credentials (`nom2`, `motdepasse2`) and invalid credentials (`nom1`, `wrong_password`).

#### Question 4: Object Diagram
A runtime snapshot of objects when `identification()` is active:
- Object `unIULogin : IULogin` with current fields `nom = "nom2"`, `motDePasse = "motdepasse2"`.
- Associated to `unGestionnaire : GestionnairePersonne`.
- Associated to instances `p1`, `p2`, `p3` of class `Personne`.

---
### 3. UML Diagrams
- Draw.io Source: [`exercise3/exercise3_diagrams.drawio`](exercise3/exercise3_diagrams.drawio)

#### Class Diagram & Object Diagram (During identification)
![Exercise 3 Class and Object Diagram](diagram_images/ex3_class_object_diagram.png)

#### Activity Diagram: IULogin Operations
![Exercise 3 Activity Diagram](diagram_images/ex3_activity_diagram.png)

In [3]:
"""
Exercise 3: User Authentication System (IULogin, GestionnairePersonne, Personne)
Tutorial 1 - UML Modeling (Software Engineering)
"""
from typing import List, Optional


# =====================================================================
# Personne Class
# =====================================================================
class Personne:
    """Represents a registered user with credentials."""

    def __init__(self, nom: str, motDePasse: str) -> None:
        self.nom = nom
        self.motDePasse = motDePasse

    def getNom(self) -> str:
        return self.nom

    def getMotDePasse(self) -> str:
        return self.motDePasse


# =====================================================================
# GestionnairePersonne Class
# =====================================================================
class GestionnairePersonne:
    """Manages the list of authorized persons."""

    def __init__(self) -> None:
        self.lesPersonnes: List[Personne] = []

    def ajoute(self, p: Personne) -> None:
        self.lesPersonnes.append(p)

    def verifierMotDePasse(self, nom: str, motDePasse: str) -> bool:
        """
        Verifies credentials against the stored persons list.
        Corresponds to loop CombinedFragment in Figure 5.
        """
        for p in self.lesPersonnes:
            n = p.getNom()
            mdp = p.getMotDePasse()
            if n.strip() == nom.strip() and mdp.strip() == motDePasse.strip():
                return True
        return False


# =====================================================================
# IULogin Class (User Interface)
# =====================================================================
class IULogin:
    """User interface handling login interactions."""

    def __init__(self) -> None:
        self.gestionnaire: Optional[GestionnairePersonne] = None
        self.nom: str = ""
        self.motDePasse: str = ""

    def setGestionnaire(self, g: GestionnairePersonne) -> None:
        self.gestionnaire = g

    def demanderNom(self, simulated_input: Optional[str] = None) -> None:
        if simulated_input is not None:
            self.nom = simulated_input
            print(f"Entrer le nom : {self.nom}")
        else:
            self.nom = input("Entrer le nom : ")

    def demanderMotDePasse(self, simulated_input: Optional[str] = None) -> None:
        if simulated_input is not None:
            self.motDePasse = simulated_input
            print(f"Entrer le mot de passe : {self.motDePasse}")
        else:
            self.motDePasse = input("Entrer le mot de passe : ")

    def autoriserConnexion(self) -> None:
        print("Connexion autorisée")

    def interdireConnexion(self) -> None:
        print("Connexion interdite")

    def identification(self, simulated_nom: Optional[str] = None, simulated_mdp: Optional[str] = None) -> None:
        """
        Full authentication procedure matching the sequence and activity diagrams.
        """
        self.demanderNom(simulated_nom)
        self.demanderMotDePasse(simulated_mdp)

        if self.gestionnaire is not None:
            res = self.gestionnaire.verifierMotDePasse(self.nom, self.motDePasse)
            if res:
                self.autoriserConnexion()
            else:
                self.interdireConnexion()
        else:
            self.interdireConnexion()


# =====================================================================
# Main execution matching tutorial setup
# =====================================================================
def main():
    unIULogin = IULogin()
    unGestionnaire = GestionnairePersonne()

    # Initialization phase
    p1 = Personne("nom1", "motdepasse1")
    p2 = Personne("nom2", "motdepasse2")
    p3 = Personne("nom3", "motdepasse3")

    unIULogin.setGestionnaire(unGestionnaire)

    unGestionnaire.ajoute(p1)
    unGestionnaire.ajoute(p2)
    unGestionnaire.ajoute(p3)

    print("=== Test 1: Successful Login (nom2, motdepasse2) ===")
    unIULogin.identification(simulated_nom="nom2", simulated_mdp="motdepasse2")

    print("\n=== Test 2: Failed Login (nom1, wrong_password) ===")
    unIULogin.identification(simulated_nom="nom1", simulated_mdp="wrong_password")


if __name__ == "__main__":
    main()

=== Test 1: Successful Login (nom2, motdepasse2) ===
Entrer le nom : nom2
Entrer le mot de passe : motdepasse2
Connexion autorisée

=== Test 2: Failed Login (nom1, wrong_password) ===
Entrer le nom : nom1
Entrer le mot de passe : wrong_password
Connexion interdite
